In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes


# Load + split your dataset (unchanged)

In [ ]:
from datasets import load_dataset

# Load your JSONL file
dataset = load_dataset("json", data_files="train.jsonl")

# Split into train/test
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 9
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 1
    })
})


# Formating dataset

In [ ]:
def format_example(example):
    return {
        "text": f"Meeting Notes:\n{example['input']}\n\nFormatted Notes:\n{example['output']}"
    }

dataset = dataset.map(format_example)


# Load falcon-rw-1b

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "tiiuae/falcon-rw-1b"

# Configure quantization for training
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 # This is important for training
)


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,   # Use the BitsAndBytesConfig
    device_map="auto"
)

In [ ]:
# def tokenize(batch):
#     return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

# tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output", "text"])


In [ ]:
# def tokenize(batch):
#     return tokenizer(
#         batch["text"],
#         truncation=True,
#         padding="max_length",
#         max_length=512
#     )

# # Apply fix before mapping
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output", "text"])


# Tokeniz

In [ ]:
def tokenize(batch):
    # Tokenize the input text and add labels
    tokenized_inputs = tokenizer(
        batch["text"],
        truncation=True,         # Ensure inputs are truncated to max_length
        padding="max_length",    # Pad sequences to max_length
        max_length=512           # Max length for the input
    )

    # Add labels for training (copy the input ids as labels for causal language modeling)
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()

    return tokenized_inputs

# Check and set pad token if necessary
# Apply fix before mapping
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# Ensure eos_token is set (important for stopping generation)
if tokenizer.eos_token is None:
    tokenizer.eos_token = "<|endoftext|>"  # Custom EOS token if needed for your model

# Apply tokenization to the dataset and remove original text columns
tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output", "text"])

In [ ]:
# def tokenize(batch):
#     # Tokenize both the input and output texts
#     input_encodings = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

#     # Set the labels to be the same as the input (for causal language modeling)
#     # The model needs to predict the labels (formatted output) from the input (meeting notes)
#     labels = tokenizer(batch["output"], truncation=True, padding="max_length", max_length=512)

#     # Set the labels for training
#     input_encodings["labels"] = labels["input_ids"]

#     return input_encodings

# # Apply fix before mapping
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# # Tokenize the dataset and add labels
# tokenized = dataset.map(tokenize, batched=True, remove_columns=["input", "output", "text"])


# Check the tokenized sample

In [ ]:
# Look at one sample
print(tokenized["train"][0])

# If you want to decode back the first input_ids to text:
print(tokenizer.decode(tokenized["train"][0]["input_ids"]))

# Check available keys
print(tokenized["train"].column_names)

# Peek at batch
for k, v in tokenized["train"][0].items():
    print(k, len(v), v[:10])  # first 10 tokens


{'input_ids': [5308, 13629, 11822, 25, 198, 36, 646, 15235, 784, 11325, 21776, 10805, 25, 220, 198, 15926, 233, 15616, 329, 406, 5653, 11812, 13, 220, 198, 15926, 233, 838, 11, 830, 3710, 2779, 13, 220, 198, 15926, 233, 370, 1187, 284, 766, 25863, 278, 13605, 13, 198, 198, 8479, 16898, 11822, 25, 198, 22093, 25, 198, 12, 8444, 29569, 406, 5653, 11812, 351, 11325, 21776, 10805, 379, 40766, 15235, 13, 198, 198, 9218, 15627, 11045, 25, 198, 12, 36557, 281, 406, 5653, 326, 460, 5412, 838, 11, 830, 2444, 13, 198, 198, 12502, 17230, 25, 198, 12, 44290, 406, 5653, 11812, 6961, 13, 198, 12, 43426, 25863, 278, 13605, 13, 198, 198, 10019, 32144, 25, 198, 12, 7281, 510, 351, 406, 5653, 13605, 290, 11812, 3307, 13, 198, 198, 10430, 14, 7575, 286, 4225, 2673, 25, 198, 12, 685, 44402, 1459, 3128, 14, 2435, 60, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 502

# LoRA config for mistral

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,            # rank
    lora_alpha=16,
    target_modules=["query_key_value"],  # depends on model, "q_proj","v_proj" for LLaMA
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


In [ ]:
# pip install -U transformers


# Initialize optimizer

In [ ]:
from torch.optim import AdamW  # Corrected import

# Initialize the optimizer
optimizer = AdamW(model.parameters(), lr=2e-4)


In [ ]:
from transformers import get_scheduler

lr_scheduler = get_scheduler(
    "linear",  # Linear scheduler (you can experiment with "cosine" if needed)
    optimizer=optimizer,
    num_warmup_steps=0,  # Optional: Learning rate warm-up
    num_training_steps=len(tokenized["train"]) * 5,  # Number of training steps (epochs * steps per epoch)
)


# training parameters

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback
import torch

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    learning_rate=2e-4,
    num_train_epochs=10,
    logging_dir="./logs",
    save_strategy="epoch",
    eval_strategy="epoch",
    metric_for_best_model="eval_loss",  # Monitor eval_loss for early stopping
    load_best_model_at_end=True,   # Automatically load the best model at the end of training
    fp16=torch.cuda.is_available(),  # mixed precision if GPU
)

# training_args = TrainingArguments(
#     output_dir="./results",
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=4,  # Simulate larger batch size
#     learning_rate=2e-4,
#     num_train_epochs=50,             # Increase to 5 epochs for better fine-tuning
#     logging_dir="./logs",
#     save_strategy="epoch",
#     eval_strategy="epoch",
#     logging_steps=10,               # Log every 10 steps
#     fp16=torch.cuda.is_available(),
#     metric_for_best_model="eval_loss",  # Monitor eval_loss for early stopping
#     load_best_model_at_end=True,   # Automatically load the best model at the end of training
# )


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized["train"],
#     eval_dataset=tokenized["test"],
#     tokenizer=tokenizer,
#     optimizers=(optimizer, lr_scheduler),
#     callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
# )

/tmp/ipython-input-2872806588.py:36: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# training process

In [ ]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,No log,0.533672
2,No log,0.483944
3,No log,0.450504
4,No log,0.423344
5,No log,0.413011
6,No log,0.421999
7,No log,0.432215


TrainOutput(global_step=35, training_loss=0.3586674281529018, metrics={'train_runtime': 21.7781, 'train_samples_per_second': 4.133, 'train_steps_per_second': 2.296, 'total_flos': 234212523245568.0, 'train_loss': 0.3586674281529018, 'epoch': 7.0})

# model saving

In [ ]:
model.save_pretrained("./lora_salesforce_notes")
tokenizer.save_pretrained("./lora_salesforce_notes")


('./lora_salesforce_notes/tokenizer_config.json',
 './lora_salesforce_notes/special_tokens_map.json',
 './lora_salesforce_notes/vocab.json',
 './lora_salesforce_notes/merges.txt',
 './lora_salesforce_notes/added_tokens.json',
 './lora_salesforce_notes/tokenizer.json')

# Check the prompt

In [ ]:
from transformers import pipeline

# Load the model pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

test_input = """MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month."""

# Create the prompt for the model
prompt = f"Meeting Notes:\n{test_input}\n\nFormatted Notes:\n"

# Generate output with eos_token_id to control where the model stops
output = pipe(prompt, max_new_tokens=200, eos_token_id=tokenizer.eos_token_id)[0]["generated_text"]

# Print the output
print(output)


Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Meeting Notes:
MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month.

Formatted Notes:
Summary:
- CFO Alex Brown discussed data security breaches and compliance with GDPR at the CIO conference.

Key Pain Points:
- Wants compliance with GDPR and compliance with HIPAA.

Action Items:
- Prepare a proposal by next month.

Next Steps:
- Review HIPAA compliance checklist.

Date/Time of Interaction:
- [Insert current date/time]


# base model output VS fine tuned model output

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel

BASE_MODEL = "tiiuae/falcon-rw-1b"
LORA_DIR = "./lora_salesforce_notes"

# Load tokenizer (shared)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# --- Base model pipeline ---
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
pipe_base = pipeline("text-generation", model=base_model, tokenizer=tokenizer, device_map="auto")

# --- LoRA model pipeline ---
lora_model = PeftModel.from_pretrained(base_model, LORA_DIR)
pipe_lora = pipeline("text-generation", model=lora_model, tokenizer=tokenizer, device_map="auto")

# --- Example input ---
test_input = """MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month."""

prompt = f"Meeting Notes:\n{test_input}\n\nFormatted Notes:\n"

# --- Generate outputs ---
out_base = pipe_base(prompt, max_new_tokens=200, eos_token_id=tokenizer.eos_token_id)[0]["generated_text"]
out_lora = pipe_lora(prompt, max_new_tokens=200, eos_token_id=tokenizer.eos_token_id)[0]["generated_text"]

# --- Show comparison ---
print("===== BASE MODEL OUTPUT =====")
print(out_base)

print("\n===== LORA MODEL OUTPUT =====")
print(out_lora)


Device set to use cuda:0
Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


===== BASE MODEL OUTPUT =====
Meeting Notes:
MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month.

Formatted Notes:
Summary:
- CIO Alex Brown discussed data security breaches and compliance with GDPR at the recent MegaTech CIO Summit.

Key Pain Points:
- Needs compliance with GDPR.

Action Items:
- Create a compliance audit plan.
- Provide a proposal by next month.

Next Steps:
- Review proposed solution.

Date/Time of Interaction:
- [Insert current date/time]

===== LORA MODEL OUTPUT =====
Meeting Notes:
MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month.

Formatted Notes:
Summary:
- Discussed data security breaches with CIO Alex Brown at MegaTech.

Key Pain Points:
- Wants compliance with GDPR.

Action Items:
- Review GDPR compliance requirements to ensure compliance.
- Provide compliance proposal by next month.

Next Ste

# Download it as a zip file

In [ ]:
from google.colab import files
!zip -r lora_salesforce_notes.zip ./lora_salesforce_notes
files.download("lora_salesforce_notes.zip")


  adding: lora_salesforce_notes/ (stored 0%)
  adding: lora_salesforce_notes/README.md (deflated 65%)
  adding: lora_salesforce_notes/special_tokens_map.json (deflated 75%)
  adding: lora_salesforce_notes/adapter_config.json (deflated 56%)
  adding: lora_salesforce_notes/adapter_model.safetensors (deflated 8%)
  adding: lora_salesforce_notes/tokenizer_config.json (deflated 52%)
  adding: lora_salesforce_notes/tokenizer.json (deflated 82%)
  adding: lora_salesforce_notes/merges.txt (deflated 53%)
  adding: lora_salesforce_notes/vocab.json (deflated 59%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_input = """MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month."""

prompt = f"Meeting Notes:\n{test_input}\n\nFormatted Notes:\n"
print(pipe(prompt, max_new_tokens=200)[0]["generated_text"])


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Meeting Notes:
MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month.

Formatted Notes:
Summary:
- Discussed data security breaches with CIO Alex Brown at MegaTech.

Key Pain Points:
- Wants compliance with GDPR.

Action Items:
- Prepare a proposal by next month.

Next Steps:
- Follow up with response.

Date/Time of Interaction:
- [Insert current date/time]
- 9.0.11.1: Updated: SQL Server 2016 Support for SQLite 3.x [KB2590187]
- 9.0.11.2: Updated: SQL Server 2016 Support for SQLite 4.x [KB2590188]
- 9.0.11.3: Updated: SQL Server 2016 Support for SQLite 5.x [KB2590201]
- 9.0.11.4: Updated: SQL Server 2016 Support for SQLite 6.x [KB2590191]
- 9.0.11.5:


In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")

# Test input: New unstructured meeting note
test_input = """MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month."""

prompt = f"Meeting Notes:\n{test_input}\n\nFormatted Notes:\n"

# Generate formatted notes
print(pipe(prompt, max_new_tokens=200)[0]["generated_text"])


Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Meeting Notes:
MegaTech – CIO Alex Brown:
○ Concerned about data security breaches.
○ Wants compliance with GDPR.
○ Requested proposal by next month.

Formatted Notes:

Presentation:

Key Messages:


Key Questions:





















-
- Page 1
- Page 1Our team is here to help you with your vehicle.
Our team is here to help you with your vehicle..Dana is the Director of Marketing and Social Media for iHeartMedia. Prior to joining iHeartMedia, Dana was Director of Marketing for the company’s digital properties. She has over a decade of experience in marketing, brand development, social media, marketing automation, and digital marketing. Dana holds a bachelor's degree in marketing from the School of Business at California Polytechnic State University, San Luis Obispo. She is a proud Cal Poly alumna. Dana lives in San Luis Obispo with her husband and two children.-
-
-
-
-
-
-
-
-
